# Dự báo Doanh số Thương mại Điện tử
## Giai đoạn 3 — Decision Tree Regressor (Core Algorithm)

> **Lưu ý phụ thuộc:** Notebook này yêu cầu đã chạy xong **GĐ1.ipynb** để có sẵn:
> - Các hàm metric: `calculate_mse`, `calculate_rmse`, `calculate_mae`, `calculate_r2`
> - Class `DanhGiaMoHinh` và object `my_evaluator`
> - Class `CustomStandardScaler` và dữ liệu `X_train_sc`, `X_test_sc`, `y_train`, `y_test`
>
> **Tầm quan trọng:** Class `DecisionTreeRegressor` này là **nền tảng bắt buộc** cho Giai đoạn 4 (Random Forest) và Giai đoạn 5 (Gradient Boosting). Nếu bước này sai, hai bước sau sẽ không chạy được.

---

## Import thư viện

In [10]:
import numpy as np
import pandas as pd

## Nhập lại từ GĐ1

Notebook này giả định đã chạy GĐ1 trong cùng kernel để có sẵn metrics và scaler.

---
## Load & Chuẩn hóa Dữ liệu

In [12]:
# Load dữ liệu đã EDA từ Phần I — BẮT BUỘC dùng pd.read_csv()
X_train = pd.read_csv("../dataset_ready/X_train.csv").values.astype(float)
X_test  = pd.read_csv("../dataset_ready/X_test.csv").values.astype(float)
y_train = pd.read_csv("../dataset_ready/y_train.csv").values.ravel().astype(float)
y_test  = pd.read_csv("../dataset_ready/y_test.csv").values.ravel().astype(float)

# Kiểm tra NaN
for name, arr in [("X_train", X_train), ("X_test", X_test),
                   ("y_train", y_train), ("y_test", y_test)]:
    assert not np.isnan(arr).any(), f"{name} có giá trị NaN!"

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")

# Chuẩn hóa — fit chỉ trên train, transform cả hai
scaler     = CustomStandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

assert np.allclose(X_train_sc.mean(axis=0), 0, atol=1e-6), "Mean không ≈ 0!"
assert np.allclose(X_train_sc.std(axis=0),  1, atol=1e-6), "Std không ≈ 1!"
print("\nDữ liệu hợp lệ và đã chuẩn hóa xong.")

X_train : (487, 9)  |  y_train : (487,)
X_test  : (122, 9)   |  y_test  : (122,)

Dữ liệu hợp lệ và đã chuẩn hóa xong.


---
## Giai đoạn 3A — Class `Node`

`Node` là cấu trúc lưu trữ của cây quyết định: feature_index, threshold, left/right, value.

In [13]:
class Node:
    """
    Đơn vị cấu trúc của Decision Tree.
    - Node nhánh: lưu feature_idx, threshold, left, right
    - Node lá   : lưu value (giá trị dự báo)
    """

    def __init__(
        self,
        feature_idx=None,
        threshold=None,
        left=None,
        right=None,
        value=None
    ):
        self.feature_idx = feature_idx  # cột đặc trưng dùng để split
        self.threshold   = threshold    # ngưỡng phân chia
        self.left        = left         # Node con bên trái  (X[:, feature] <= threshold)
        self.right       = right        # Node con bên phải  (X[:, feature] >  threshold)
        self.value       = value        # giá trị lá (mean của y); None nếu là node nhánh

    def is_leaf(self):
        """Trả về True nếu đây là node lá (không có con)"""
        return self.value is not None


print("Đã định nghĩa xong class Node.")

Đã định nghĩa xong class Node.


In [14]:
# ── Kiểm tra nhanh Node ──────────────────────────────────────────────────────
leaf_node   = Node(value=42.0)
branch_node = Node(feature_idx=2, threshold=0.5)

assert leaf_node.is_leaf()   == True,  "Node lá phải trả về is_leaf() = True"
assert branch_node.is_leaf() == False, "Node nhánh phải trả về is_leaf() = False"
assert leaf_node.value       == 42.0,  "Node lá phải lưu đúng value"
assert branch_node.feature_idx == 2,   "Node nhánh phải lưu đúng feature_idx"
assert branch_node.threshold   == 0.5, "Node nhánh phải lưu đúng threshold"

print("Tất cả assert Node PASSED.")

Tất cả assert Node PASSED.


---
## Giai đoạn 3B — Class `DecisionTreeRegressor`

Những giải thích chi tiết về cơ chế xây dựng cây đã được cô đọng trong docstring của class. Phần này tập trung vào code và đánh giá thực nghiệm.

In [15]:
class DecisionTreeRegressor:
    """Decision Tree Regressor implemented từ đầu bằng NumPy."""

    def __init__(
        self,
        max_depth=5,
        min_samples_split=10,
        max_features=None,
        evaluator=None
    ):
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.max_features      = max_features
        self.evaluator         = evaluator  # inject từ GĐ1
        self.root              = None       # gốc cây, được gán sau fit()

    # ──────────────────────────────────────────────────────────────────────────
    # PRIVATE: Tính Variance Reduction của một phép chia
    # ──────────────────────────────────────────────────────────────────────────
    def _calculate_variance_reduction(self, y_parent, y_left, y_right):
        """
        Đánh giá chất lượng của một phép chia bằng Variance Reduction.

        VR = Var(y_parent) - (n_L/n)*Var(y_L) - (n_R/n)*Var(y_R)

        Giá trị VR càng lớn → phép chia càng tốt.

        Returns
        -------
        float — mức độ giảm phương sai
        """
        n   = len(y_parent)
        n_L = len(y_left)
        n_R = len(y_right)

        # Nếu một trong hai nhánh rỗng → phép chia vô nghĩa
        if n_L == 0 or n_R == 0:
            return 0.0

        var_parent = np.var(y_parent)
        var_left   = np.var(y_left)
        var_right  = np.var(y_right)

        return var_parent - (n_L / n) * var_left - (n_R / n) * var_right

    # ──────────────────────────────────────────────────────────────────────────
    # PRIVATE: Tìm phép chia tối ưu (Exhaustive Search)
    # ──────────────────────────────────────────────────────────────────────────
    def _get_best_split(self, X, y):
        """
        Duyệt toàn bộ features và threshold để tìm split tối đa hóa VR.

        - Nếu max_features được đặt → lấy mẫu ngẫu nhiên max_features cột
          (dùng cho Random Forest để tăng diversity giữa các cây).
        - Threshold ứng viên = tập hợp các giá trị duy nhất của feature đó.

        Returns
        -------
        dict: {'feature_idx': int, 'threshold': float, 'vr': float}
              hoặc None nếu không tìm được split tốt hơn baseline (VR=0)
        """
        n_features   = X.shape[1]
        best_vr      = 0.0          # chỉ chấp nhận split có VR > 0
        best_split   = None

        # Chọn tập feature cần xét
        if self.max_features is not None:
            k = min(self.max_features, n_features)
            feature_indices = np.random.choice(n_features, k, replace=False)
        else:
            feature_indices = np.arange(n_features)

        for feat_idx in feature_indices:
            thresholds = np.unique(X[:, feat_idx])

            for threshold in thresholds:
                mask_left  = X[:, feat_idx] <= threshold
                mask_right = ~mask_left

                y_left  = y[mask_left]
                y_right = y[mask_right]

                # Bỏ qua nếu một nhánh rỗng
                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                vr = self._calculate_variance_reduction(y, y_left, y_right)

                if vr > best_vr:
                    best_vr    = vr
                    best_split = {
                        "feature_idx": feat_idx,
                        "threshold":   threshold,
                        "vr":          vr,
                    }

        return best_split

    # ──────────────────────────────────────────────────────────────────────────
    # PRIVATE: Đệ quy xây dựng cây
    # ──────────────────────────────────────────────────────────────────────────
    def _build_tree(self, X, y, depth=0):
        """
        Đệ quy xây dựng cấu trúc cây từ gốc xuống lá.

        Điều kiện dừng (tạo node lá):
          1. Đạt max_depth
          2. Số mẫu < min_samples_split
          3. Không tìm được split tốt hơn (best_split = None)
          4. Tất cả y đã giống nhau (var = 0)

        Returns
        -------
        Node
        """
        n_samples = len(y)

        # ── Kiểm tra điều kiện dừng ──────────────────────────────────────────
        if (
            depth >= self.max_depth
            or n_samples < self.min_samples_split
            or np.var(y) == 0
        ):
            leaf_value = np.mean(y)   # giá trị dự báo = trung bình y trong vùng
            return Node(value=leaf_value)

        # ── Tìm phép chia tốt nhất ───────────────────────────────────────────
        best_split = self._get_best_split(X, y)

        if best_split is None:
            # Không còn split nào cải thiện được VR → tạo lá
            return Node(value=np.mean(y))

        feat_idx  = best_split["feature_idx"]
        threshold = best_split["threshold"]

        mask_left  = X[:, feat_idx] <= threshold
        mask_right = ~mask_left

        # ── Đệ quy xây nhánh con ─────────────────────────────────────────────
        left_child  = self._build_tree(X[mask_left],  y[mask_left],  depth + 1)
        right_child = self._build_tree(X[mask_right], y[mask_right], depth + 1)

        return Node(
            feature_idx=feat_idx,
            threshold=threshold,
            left=left_child,
            right=right_child,
        )

    # ──────────────────────────────────────────────────────────────────────────
    # PRIVATE: Đệ quy duyệt cây cho một mẫu
    # ──────────────────────────────────────────────────────────────────────────
    def _traverse_tree(self, x, node):
        """
        Đi từ gốc xuống lá để dự báo một mẫu x.

        Parameters
        ----------
        x    : array (n_features,) — một hàng dữ liệu
        node : Node — bắt đầu từ self.root

        Returns
        -------
        float — giá trị dự báo tại node lá
        """
        if node.is_leaf():
            return node.value

        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC: Huấn luyện
    # ──────────────────────────────────────────────────────────────────────────
    def fit(self, X, y):
        """
        Xây dựng cấu trúc cây từ dữ liệu huấn luyện.

        Parameters
        ----------
        X : array (n_samples, n_features)
        y : array (n_samples,)

        Returns
        -------
        self
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        self.root = self._build_tree(X, y, depth=0)
        return self

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC: Dự báo
    # ──────────────────────────────────────────────────────────────────────────
    def predict(self, X):
        """
        Dự báo trên toàn bộ tập dữ liệu bằng cách duyệt cây cho từng hàng.

        Parameters
        ----------
        X : array (n_samples, n_features)

        Returns
        -------
        y_pred : array (n_samples,)
        """
        if self.root is None:
            raise RuntimeError("Cần gọi fit() trước khi predict()")
        X = np.asarray(X, dtype=float)
        return np.array([self._traverse_tree(x, self.root) for x in X])

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC: Đánh giá (gọi evaluator từ Bước 1)
    # ──────────────────────────────────────────────────────────────────────────
    def evaluate(self, X_test, y_test, model_name="Decision Tree"):
        """
        Dự báo và đánh giá — gọi DanhGiaMoHinh từ Bước 1, không tự tính metric.

        Returns
        -------
        dict: {'MSE', 'RMSE', 'MAE', 'R2'}
        """
        if self.evaluator is None:
            raise RuntimeError("evaluator chưa được inject. Truyền DanhGiaMoHinh() vào __init__.")
        y_pred = self.predict(X_test)
        return self.evaluator.evaluate_all(y_test, y_pred, model_name=model_name)


print("Đã định nghĩa xong class DecisionTreeRegressor.")

Đã định nghĩa xong class DecisionTreeRegressor.


---
## Kiểm tra nhanh với dữ liệu đơn giản

Dùng bài toán đồ chơi có thể tính tay để xác minh cây hoạt động đúng trước khi chạy dữ liệu thực.

In [16]:
# ── Bài toán đồ chơi: 1 feature, dữ liệu tách biệt rõ ràng ──────────────────
#   X < 5 → y ≈ 10 | X >= 5 → y ≈ 20
#   Cây depth=1 phải tìm được split tại threshold=4 hoặc 5 với VR cực đại.

X_toy = np.array([[1],[2],[3],[4],[6],[7],[8],[9]], dtype=float)
y_toy = np.array([10, 10, 10, 10, 20, 20, 20, 20], dtype=float)

toy_tree = DecisionTreeRegressor(
    max_depth=3,
    min_samples_split=2,
    evaluator=my_evaluator
)
toy_tree.fit(X_toy, y_toy)

y_toy_pred = toy_tree.predict(X_toy)

# Dữ liệu tách biệt hoàn toàn → R² phải = 1.0
r2_toy = calculate_r2(y_toy, y_toy_pred)
assert abs(r2_toy - 1.0) < 1e-6, f"R² đồ chơi phải = 1.0, nhận được: {r2_toy:.6f}"

# Root phải là node nhánh (không phải lá)
assert not toy_tree.root.is_leaf(), "Root phải là node nhánh!"

# Dự báo phải đúng giá trị nhóm
assert y_toy_pred[0] == 10.0, f"Nhóm trái phải dự báo 10.0, nhận: {y_toy_pred[0]}"
assert y_toy_pred[4] == 20.0, f"Nhóm phải phải dự báo 20.0, nhận: {y_toy_pred[4]}"

print(f"R² đồ chơi : {r2_toy:.4f}")
print(f"Split tại  : feature={toy_tree.root.feature_idx}, threshold={toy_tree.root.threshold}")
print("Tất cả assert kiểm tra nhanh PASSED.")

R² đồ chơi : 1.0000
Split tại  : feature=0, threshold=4.0
Tất cả assert kiểm tra nhanh PASSED.


---
## Chạy trên Dữ liệu Thực tế

Thử nghiệm với nhiều bộ hyperparameter để quan sát ảnh hưởng của `max_depth`.

In [17]:
# ── Khảo sát ảnh hưởng của max_depth ─────────────────────────────────────────
print("── Khảo sát max_depth ──")
print(f"{'Depth':>6} | {'R² Train':>10} | {'R² Test':>10} | {'RMSE Test':>12}")
print("-" * 46)

depth_results = {}
for depth in [3, 5, 7, 10]:
    dt = DecisionTreeRegressor(
        max_depth=depth,
        min_samples_split=10,
        evaluator=my_evaluator
    )
    dt.fit(X_train_sc, y_train)

    r2_train = calculate_r2(y_train, dt.predict(X_train_sc))
    r2_test  = calculate_r2(y_test,  dt.predict(X_test_sc))
    rmse_test = calculate_rmse(y_test, dt.predict(X_test_sc))

    depth_results[depth] = {"r2_train": r2_train, "r2_test": r2_test, "rmse": rmse_test}
    print(f"{depth:>6} | {r2_train:>10.4f} | {r2_test:>10.4f} | {rmse_test:>12.4f}")

── Khảo sát max_depth ──
 Depth |   R² Train |    R² Test |    RMSE Test
----------------------------------------------
     3 |     0.6812 |     0.3144 |    9307.8945
     5 |     0.7469 |     0.5024 |    7929.6527
     7 |     0.7919 |     0.5379 |    7641.7574
    10 |     0.8127 |     0.5497 |    7543.1196


In [18]:
# ── Huấn luyện và đánh giá chính thức với depth=5 ────────────────────────────
# depth=5 thường là điểm cân bằng bias-variance tốt cho dataset vừa nhỏ
print("=" * 50)
dt_model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=10,
    evaluator=my_evaluator
)
dt_model.fit(X_train_sc, y_train)
dt_results = dt_model.evaluate(X_test_sc, y_test, model_name="Decision Tree (depth=5)")

# Sanity check
assert dt_results["R2"] > 0, f"R² âm — mô hình tệ hơn baseline mean: {dt_results['R2']:.4f}"
assert dt_model.root is not None, "Cây chưa được xây dựng!"
assert not dt_model.root.is_leaf(), "Root không được là lá!"
print(f"\nSanity check R² > 0: PASSED ({dt_results['R2']:.4f})")
print("\ndt_model sẵn sàng để tái sử dụng ở Giai đoạn 4 và Giai đoạn 5.")

── Đánh giá Decision Tree (depth=5) ──
MSE  : 62879391.9263
RMSE : 7929.6527
MAE  : 6036.6080
R²   : 0.5024

Sanity check R² > 0: PASSED (0.5024)

dt_model sẵn sàng để tái sử dụng ở Giai đoạn 4 và Giai đoạn 5.
